<a href="https://colab.research.google.com/github/pashapara22/-pashapara22-word-guess-games/blob/main/Tool_Use%26Func_Calling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install groq
from google.colab import userdata
from groq import Groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 138.3/138.3 kB 4.2 MB/s eta 0:00:00


In [2]:
import requests
import json

In [3]:
from pprint import pprint

In [4]:
def get_weather(location):
  api_key = userdata.get('weather_api')
  url = f"  http://api.openweathermap.org/data/2.5/weather?q={location}&units=metric&appid={api_key}"
  response = requests.get(url)
  data = response.json()

  if data['cod'] ==200:
    return{
        "location":location,
        "temperature": data['main']['temp']
    }
  else:
    return{
        "Message":"Location not found"
    }

In [5]:
tools = [
  {
    "type": "function",
    "function": {
      "name": "get_weather",
      "description": "Get current weather for a city",
      "parameters": {
        "type": "object",
        "properties": {
          "location": {
            "type": "string",
            "description": "City name like Mumbai, London"
          }},
        "required": ["location"]
      }}}
]

In [6]:
output = get_weather("Hyderabad")
pprint(output)

{'location': 'Hyderabad', 'temperature': 34.23}


In [7]:
llm_messages = [
  {
    "role": "system",
    "content": "You are a weather assistant. Use get_weather function when asked about weather."
  },
  {
    "role": "user",
    "content": "What's the weather in Hyderabad?"
  }
]

In [8]:
client = Groq(
  api_key=userdata.get('GROQ_API')
)

response = client.chat.completions.create(
  messages=llm_messages,
  model="llama-3.3-70b-versatile",
  tools = tools,
  tool_choice = "auto"
)


In [9]:
#print(response.choices[0].message)
response_call =response
print(response.model_dump_json(indent=3))



{
   "id": "chatcmpl-1515b9f7-2e33-4a5f-9fea-60fa950f46bc",
   "choices": [
      {
         "finish_reason": "tool_calls",
         "index": 0,
         "logprobs": null,
         "message": {
            "content": null,
            "role": "assistant",
            "annotations": null,
            "executed_tools": null,
            "function_call": null,
            "reasoning": null,
            "tool_calls": [
               {
                  "id": "0kqdv3h38",
                  "function": {
                     "arguments": "{\"location\":\"Hyderabad\"}",
                     "name": "get_weather"
                  },
                  "type": "function"
               }
            ]
         }
      }
   ],
   "created": 1772780081,
   "model": "llama-3.3-70b-versatile",
   "object": "chat.completion",
   "mcp_list_tools": null,
   "service_tier": "on_demand",
   "system_fingerprint": "fp_3272ea2d91",
   "usage": {
      "completion_tokens": 15,
      "prompt_tokens": 243,
 

In [14]:
response_call = response.choices[0].message

if response_call.tool_calls:
  tool_call = response_call.tool_calls[0]
  arguments = json.loads(tool_call.function.arguments)
  location = arguments['location']
  weather_Data = get_weather(location)
  #print(weather_Data)

  llm_messages.append(response_call)

  llm_messages.append({
      "role": "tool",
      "tool_call_id":tool_call.id,
      "content": str(weather_Data)
  })

  final_response = client.chat.completions.create(
  messages=llm_messages,
  model="llama-3.3-70b-versatile")

  print(final_response.choices[0].message.content)

The current weather in Hyderabad is 34.23 degrees Celsius.
